In [6]:
import pandas as pd
import numpy as np
!pip install scikeras

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

from scikeras.wrappers import KerasClassifier

In [8]:
#Loading datasets
data = pd.read_csv("mobile_price_classification.csv")

data.head()

,battery_power,bluetooth,clock_speed,dual_sim,front_cam,4G,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1


In [9]:
#defining features and target
X = data.drop("price_range", axis=1)
y = data["price_range"]

In [10]:
#train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
#feature scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [12]:
#ANN model creation
def create_model(optimizer='adam'):

    model = Sequential()

    model.add(Dense(32, activation='relu', input_dim=X_train.shape[1]))
    model.add(Dense(16, activation='relu'))

    model.add(Dense(4, activation='softmax'))

    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [13]:
#wrapping model
model = KerasClassifier(model=create_model, verbose=0)

In [25]:
#hyperparametric tuning
optimizers = ["adam", "rmsprop"]
batch_sizes = [16, 32]
epochs_list = [50, 100]

best_accuracy = 0
best_model = None
best_params = {}

for opt in optimizers:
    for batch in batch_sizes:
        for ep in epochs_list:

            model = create_model(optimizer=opt)

            model.fit(
                X_train, y_train,
                epochs=ep,
                batch_size=batch,
                verbose=0
            )

            loss, acc = model.evaluate(X_test, y_test, verbose=0)

            if acc > best_accuracy:
                best_accuracy = acc
                best_model = model
                best_params = {"optimizer": opt, "batch_size": batch, "epochs": ep}

print("Best Parameters:", best_params)
print("Best Accuracy:", best_accuracy)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best Parameters: {'optimizer': 'adam', 'batch_size': 16, 'epochs': 50}
Best Accuracy: 0.9424999952316284


In [26]:
#prediction
y_pred = best_model.predict(X_test)
y_pred = y_pred.argmax(axis=1)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 


In [27]:
#model evaluation
print("Test Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Test Accuracy: 0.9425

Classification Report:

              precision    recall  f1-score   support

           0       0.95      0.95      0.95       105
           1       0.91      0.95      0.92        91
           2       0.96      0.89      0.93        92
           3       0.95      0.97      0.96       112

    accuracy                           0.94       400
   macro avg       0.94      0.94      0.94       400
weighted avg       0.94      0.94      0.94       400

